In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import copy
from scipy.stats import wilcoxon, mannwhitneyu
import pickle as pkl

import nibabel as nib

import matplotlib.pyplot as plt

In [2]:
def load_unet_result(path, _print):
    unet_df = pd.read_csv(path, index_col = 'Unnamed: 0')
    index_values = []
    for _index in unet_df.index:
        index_values.append(_index.split('-seg')[0])

    unet_df.index = index_values  
    unet_df.drop(['WT jaccard', 'TC jaccard', 'ET jaccard'], axis = 1, inplace = True)
    summary_unet_df = pd.DataFrame(zip(unet_df.mean().values.tolist(), 
                                       unet_df.std().values.tolist()), 
                                   columns = ['mean', 'std'], index = unet_df.columns)
    if _print:
        print("****UNet******")
        print(summary_unet_df)
    return summary_unet_df, unet_df

In [3]:
# Load the JSON file
def load_nnunet_result(path, _print):
    with open(path, 'r') as file:
        data = json.load(file)

    WT = []
    TC = []
    ET = []
    file_name = []
    for case in data['metric_per_case']:
        WT.append(case['metrics']['(2, 1, 3)']['Dice'])
        TC.append(case['metrics']['(2, 3)']['Dice'])
        ET.append(case['metrics']['(3,)']['Dice'])
        file_name.append(case['reference_file'].split('/')[-1].split('.')[0])

    nnunet_df = pd.DataFrame(zip(WT, TC, ET), columns = ['WT dice', 'TC dice', 'ET dice'], 
                             index = file_name)
    summary_nnunet_df = pd.DataFrame(zip(nnunet_df.mean().values.tolist(), 
                                       nnunet_df.std().values.tolist()), 
                                   columns = ['mean', 'std'], index = nnunet_df.columns)
    if _print:
        print("****nnUNet******")
        print(summary_nnunet_df)
    return summary_nnunet_df, nnunet_df

In [4]:
def load_TransBTS_result(path, _print):
    with open(path, 'r') as file:
        data = json.load(file)

    WT = []
    TC = []
    ET = []
    file_name = []
    for case_id in data.keys():
        case = data[case_id]
        WT.append(case['WT'][0])
        TC.append(case['TC'][0])
        ET.append(case['ET'][0])
        file_name.append(case_id)

    TransBTS_df = pd.DataFrame(zip(WT, TC, ET), columns = ['WT dice', 'TC dice', 'ET dice'], 
                             index = file_name)
    summary_TransBTS_df = pd.DataFrame(zip(TransBTS_df.mean().values.tolist(), 
                                       TransBTS_df.std().values.tolist()), 
                                   columns = ['mean', 'std'], index = TransBTS_df.columns)
    if _print:
        print("****TransBTS******")
        print(summary_TransBTS_df)
    return summary_TransBTS_df, TransBTS_df

In [5]:
def read_results(_print=True):
    path = '../Results/Result/Vanilla_Unet/Unet_test_dice.csv'
    summary_unet_df, unet_df = load_unet_result(path, _print)

    path = '../Results/Result/nnUnet/nnUNetTrainer/summary.json'
    summary_da_nnunet_df, nnunet_da_df = load_nnunet_result(path, _print)

    path = '../Results/Result/nnUnet/nnUNetTrainerNoDA/summary.json'
    summary_noda_nnunet_df, nnunet_noda_df = load_nnunet_result(path, _print)

    path = '../Results/Result/TransBTS/submission/TransBTS2023-11-03/TransBTS_summary.json'
    summary_TransBTS_df, TransBTS_df = load_TransBTS_result(path, _print)
    return summary_unet_df, unet_df, summary_noda_nnunet_df, nnunet_noda_df, summary_da_nnunet_df, nnunet_da_df, summary_TransBTS_df, TransBTS_df

In [6]:
_ = read_results()


****UNet******
             mean       std
WT dice  0.888850  0.131130
TC dice  0.826315  0.214695
ET dice  0.783685  0.208260
****nnUNet******
             mean       std
WT dice  0.951790  0.084795
TC dice  0.930236  0.138221
ET dice  0.892424  0.149592
****nnUNet******
             mean       std
WT dice  0.940657  0.103507
TC dice  0.909993  0.166380
ET dice  0.874730  0.166799
****TransBTS******
             mean       std
WT dice  0.894785  0.132437
TC dice  0.871829  0.197529
ET dice  0.828126  0.201881


In [7]:
def get_overlaps(unet_df, TransBTS_df, nnunet_noda_df, dice_threshold, dice_score):
    
    WT_dice_threshold= 0.91
    TC_dice_threshold= 0.86
    ET_dice_threshold= 0.85
    
    unet_df_sub = unet_df[(unet_df['WT dice'] < WT_dice_threshold) 
                          & (unet_df['TC dice'] < TC_dice_threshold) 
                          & (unet_df['ET dice'] < ET_dice_threshold)]
    
    TransBTS_df_sub = TransBTS_df[(TransBTS_df['WT dice'] < WT_dice_threshold) 
                          & (TransBTS_df['TC dice'] < TC_dice_threshold) 
                          & (TransBTS_df['ET dice'] < ET_dice_threshold)]
    
    nnunet_noda_df_sub = nnunet_noda_df[(nnunet_noda_df['WT dice'] < WT_dice_threshold) 
                          & (nnunet_noda_df['TC dice'] < TC_dice_threshold) 
                          & (nnunet_noda_df['ET dice'] < ET_dice_threshold)]

    unet_df_sub_subjects = unet_df_sub.index.values.tolist()
    TransBTS_df_sub_subjects = TransBTS_df_sub.index.values.tolist()
    nnunet_noda_df_sub_subjects = nnunet_noda_df_sub.index.values.tolist()

    all_overlaps = list(set(unet_df_sub_subjects) & set(TransBTS_df_sub_subjects) & set(nnunet_noda_df_sub_subjects))
    print('all overlap', len(all_overlaps), unet_df_sub.shape, nnunet_noda_df_sub.shape)

    unet_nnunet_overlaps = list(set(unet_df_sub_subjects) & set(nnunet_noda_df_sub_subjects))
    print('unet-nnunet overlap', len(unet_nnunet_overlaps))

    unet_TransBTS_overlaps = list(set(unet_df_sub_subjects) & set(TransBTS_df_sub_subjects))
    print('unet-TransBTS overlap', len(unet_TransBTS_overlaps))

    nnunet_TransBTS_overlaps = list(set(TransBTS_df_sub_subjects) & set(nnunet_noda_df_sub_subjects))
    print('nnunet-TransBTS overlap', len(nnunet_TransBTS_overlaps))
    
    return all_overlaps, unet_nnunet_overlaps, unet_TransBTS_overlaps, nnunet_TransBTS_overlaps


In [8]:
def read_radiomics_results(analysis_type, location):
    file_name = '../Results/Analysis_Results/Radiomics/' + location + '/' + analysis_type + '.pkl'
    with open(file_name, 'rb') as f:
        results = pkl.load(f)
    return results

def read_MRI(dataset, patient_id):
    if dataset == 'Brats2020':
        baseloc = '../input/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData/'
        pefix = 'BraTS20_Training_' + patient_id + '/' + 'BraTS20_Training_' + patient_id
        suffixs = ['_flair.nii','_t2.nii', '_t1.nii', '_t1ce.nii', '_seg.nii']
    elif dataset == 'Brats2023':
        baseloc = '../input/Brats2023/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData/'
        pefix = patient_id + '/' + patient_id
        suffixs = ['-t2f.nii.gz','-t2w.nii.gz', '-t1n.nii.gz', '-t1c.nii.gz', '-seg.nii.gz']

    flair_filename = baseloc + pefix + suffixs[0]
    flair_img_f = nib.load(flair_filename)
    flair_img = np.asarray(flair_img_f.dataobj)

    t2_filename = baseloc + pefix + suffixs[1]
    t2_img_f = nib.load(t2_filename)
    t2_img = np.asarray(t2_img_f.dataobj)

    t1_filename = baseloc + pefix + suffixs[2]
    t1_img_f = nib.load(t1_filename)
    t1_img = np.asarray(t1_img_f.dataobj)

    t1ce_filename = baseloc + pefix + suffixs[3]
    t1ce_img_f = nib.load(t1ce_filename)
    t1ce_img = np.asarray(t1ce_img_f.dataobj)
    
    mask_filename = baseloc + pefix + suffixs[4]
    mask_img_f = nib.load(mask_filename)
    mask_img = np.asarray(mask_img_f.dataobj)
    
    return flair_img, t2_img, t1_img, t1ce_img, mask_img 

def preprocess_mask_labels(mask):
    # whole tumour
    mask_WT = mask.copy()
    mask_WT[mask_WT == 1] = 1
    mask_WT[mask_WT == 2] = 1
    mask_WT[mask_WT == 3] = 1
    # include all tumours 

    # NCR / NET - LABEL 1
    mask_TC = mask.copy()
    mask_TC[mask_TC == 1] = 1
    mask_TC[mask_TC == 2] = 0
    mask_TC[mask_TC == 3] = 1
    # exclude 2 / 4 labelled tumour 

    # ET - LABEL 4 
    mask_ET = mask.copy()
    mask_ET[mask_ET == 1] = 0
    mask_ET[mask_ET == 2] = 0
    mask_ET[mask_ET == 3] = 1
    # exclude 2 / 1 labelled tumour 

    mask = np.stack([mask_WT, mask_TC, mask_ET])
    
    return mask 

def read_mask_file(patient_id):
    baseloc = '../input/Brats2023/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData/'
    pefix = patient_id + '/' + patient_id
    suffixs = ['-t2f.nii.gz','-t2w.nii.gz', '-t1n.nii.gz', '-t1c.nii.gz', '-seg.nii.gz']
    outputloc = '../Results/Result/Vanilla_Unet/' + patient_id
    
    sample_filename_mask = baseloc + pefix + suffixs[4]
    sample_mask_f = nib.load(sample_filename_mask)
    sample_mask = np.asarray(sample_mask_f.dataobj)
    
    masks = preprocess_mask_labels(sample_mask)
    mask_WT, mask_TC, mask_ET = masks[0], masks[1], masks[2]
    
    return mask_WT

def cliffs_delta(x, y):
    n_x = len(x)
    n_y = len(y)
    N_gr = sum(xi > yi for xi in x for yi in y)
    N_ls = sum(xi < yi for xi in x for yi in y)
    return (N_gr - N_ls) / (n_x * n_y)

def cohens_d(group1, group2):
    """
    Calculate Cohen's d for measuring effect size between two groups.

    Parameters:
    - group1, group2: Arrays or lists of observations for the two groups.

    Returns:
    - d: Cohen's d (effect size).
    """
    # Calculate the means of the two groups
    mean1, mean2 = np.mean(group1), np.mean(group2)
    
    # Calculate the pooled standard deviation
    # (N1 - 1) * var1 + (N2 - 1) * var2 / (N1 + N2 - 2)
    n1, n2 = len(group1), len(group2)
    var1, var2 = np.var(group1, ddof=1), np.var(group2, ddof=1)
    pooled_std = np.sqrt(((n1 - 1) * var1 + (n2 - 1) * var2) / (n1 + n2 - 2))
    
    # Calculate Cohen's d
    d = (mean1 - mean2) / pooled_std
    
    return d

In [9]:
def analyze_result(overlaps, unet_df, property_result_df):    
    
    unet_df = pd.merge(unet_df, 
                       property_result_df, 
                       left_index=True, 
                       right_index=True)
    WT_dice_threshold= 0.91
    TC_dice_threshold= 0.86
    ET_dice_threshold= 0.85
    

    unet_df_bad = unet_df[(unet_df['WT dice'] < WT_dice_threshold) 
                          & (unet_df['TC dice'] < TC_dice_threshold) 
                          & (unet_df['ET dice'] < ET_dice_threshold)]
    unet_df_bad = unet_df_bad[unet_df_bad.index.isin(overlaps)]

    unet_df_good = unet_df[(unet_df['WT dice'] >= WT_dice_threshold) 
                           & (unet_df['TC dice'] >= TC_dice_threshold)]
    
    result_df = pd.DataFrame(zip(unet_df_bad.median().round(2), unet_df_good.median().round(2)), 
                                 index = unet_df_bad.mean().index, 
                                 columns = ['bad_seg', 'good_seg'])
    result_df['Statistically_Different'] = ['-']*result_df.shape[0]
    result_df['Effect_value'] = [0]*result_df.shape[0]
    result_df['Effect_Size'] = ['No']*result_df.shape[0]
    

    for _index in result_df.index:
        res, p = mannwhitneyu(unet_df_bad[_index].values, 
                              unet_df_good[_index].values, 
                              alternative='two-sided', 
                              method='asymptotic')

        size = round(abs(cliffs_delta(unet_df_bad[_index].values, 
                          unet_df_good[_index].values)), 2)
        
        if p < 0.001:
            result_df.loc[_index, 'Statistically_Different'] = '***'
        elif p < 0.01:
            result_df.loc[_index, 'Statistically_Different'] = '**'
        elif p < 0.05:
            result_df.loc[_index, 'Statistically_Different'] = '*'
            
        result_df.loc[_index, 'Effect_value'] = round(size, 2)
        
        if size < 0.147:
            result_df.loc[_index, 'Effect_Size'] = 'No'
        elif (size >= 0.147) & (size < 0.33):
            result_df.loc[_index, 'Effect_Size'] = 'Small'
        elif(size >= 0.33) & (size < 0.47):
            result_df.loc[_index, 'Effect_Size'] = 'Medium'
        elif size >= 0.47:
            result_df.loc[_index, 'Effect_Size'] = 'Large'
        else:
            print('Error')
            
    return result_df

In [21]:
dice_threshold = 0.91
dice_score = 'WT dice'
location = 'Tumor_WT'
analysis_types = ['shape', 
                  'size' ]

summary_unet_df, unet_df, summary_noda_nnunet_df, nnunet_noda_df, summary_da_nnunet_df, nnunet_da_df, summary_TransBTS_df, TransBTS_df = read_results(False)

all_overlaps, unet_nnunet_overlaps, unet_TransBTS_overlaps, nnunet_TransBTS_overlaps = get_overlaps(unet_df,
                                                                                        TransBTS_df,
                                                                                        nnunet_noda_df,
                                                                                        dice_threshold, 
                                                                                       dice_score)
overlaps = unet_nnunet_overlaps
print(len(overlaps))


all overlap 42 (157, 3) (47, 3)
unet-nnunet overlap 47
unet-TransBTS overlap 93
nnunet-TransBTS overlap 42
47


{'BraTS-GLI-00805-000': {'flair': 85169.0,
  't2': 85169.0,
  't1': 85169.0,
  't1ce': 85169.0},
 'BraTS-GLI-01496-000': {'flair': 218263.0,
  't2': 218263.0,
  't1': 218263.0,
  't1ce': 218263.0},
 'BraTS-GLI-01486-000': {'flair': 143067.0,
  't2': 143067.0,
  't1': 143067.0,
  't1ce': 143067.0},
 'BraTS-GLI-00645-001': {'flair': 20044.0,
  't2': 20044.0,
  't1': 20044.0,
  't1ce': 20044.0},
 'BraTS-GLI-00655-001': {'flair': 148517.0,
  't2': 148517.0,
  't1': 148517.0,
  't1ce': 148517.0},
 'BraTS-GLI-00382-000': {'flair': 53436.0,
  't2': 53436.0,
  't1': 53436.0,
  't1ce': 53436.0},
 'BraTS-GLI-00392-000': {'flair': 47590.0,
  't2': 47590.0,
  't1': 47590.0,
  't1ce': 47590.0},
 'BraTS-GLI-01064-000': {'flair': 33518.0,
  't2': 33518.0,
  't1': 33518.0,
  't1ce': 33518.0},
 'BraTS-GLI-00416-000': {'flair': 206933.0,
  't2': 206933.0,
  't1': 206933.0,
  't1ce': 206933.0},
 'BraTS-GLI-01074-000': {'flair': 95372.0,
  't2': 95372.0,
  't1': 95372.0,
  't1ce': 95372.0},
 'BraTS-GLI-00

In [30]:
location = 'Tumor_WT'

WT_df = read_radiomics_results('size', location)
WT_df = pd.DataFrame.from_dict(WT_df['original_shape_VoxelVolume'], orient='index')
WT_df.drop(['t2', 't1', 't1ce'], axis = 1, inplace = True)
WT_df.columns = ['WT']

location = 'Tumor_TC'

TC_df = read_radiomics_results('size', location)
TC_df = pd.DataFrame.from_dict(TC_df['original_shape_VoxelVolume'], orient='index')
TC_df.drop(['t2', 't1', 't1ce'], axis = 1, inplace = True)
TC_df.columns = ['TC']

location = 'Tumor_ET'

ET_df = read_radiomics_results('size', location)
ET_df = pd.DataFrame.from_dict(ET_df['original_shape_VoxelVolume'], orient='index')
ET_df.drop(['t2', 't1', 't1ce'], axis = 1, inplace = True)
ET_df.columns = ['ET']

In [37]:
df = pd.merge(WT_df, 
               TC_df, 
               left_index=True, 
               right_index=True)

df = pd.merge(df, 
               ET_df, 
               left_index=True, 
               right_index=True)
df['TC_WT'] = df['TC']/df['WT']
df['ET_WT'] = df['ET']/df['WT']
df['ET_TC'] = df['ET']/df['TC']

In [38]:
result_df = analyze_result(overlaps, unet_df, df)

In [41]:
df[df['ET_TC'] < 0.5]

,WT,TC,ET,TC_WT,ET_WT,ET_TC
BraTS-GLI-01496-000,218263.0,57282.0,126.0,0.262445,0.000577,0.002200
BraTS-GLI-00382-000,53436.0,25761.0,10532.0,0.482091,0.197096,0.408835
BraTS-GLI-00231-000,79221.0,19123.0,9305.0,0.241388,0.117456,0.486587
BraTS-GLI-01410-000,93395.0,81497.0,24896.0,0.872606,0.266567,0.305484
BraTS-GLI-01245-000,91646.0,17190.0,6708.0,0.187570,0.073195,0.390227
...,...,...,...,...,...,...
BraTS-GLI-01493-000,77001.0,37156.0,18283.0,0.482539,0.237438,0.492061
BraTS-GLI-00130-000,149604.0,62554.0,26937.0,0.418131,0.180055,0.430620
BraTS-GLI-01003-000,127812.0,82014.0,33404.0,0.641677,0.261353,0.407296
BraTS-GLI-01458-000,60356.0,39093.0,17097.0,0.647707,0.283269,0.437342


In [13]:
all_df = pd.DataFrame()

for analysis_type in analysis_types:
    try:
        radiomics_results_df = read_radiomics_results(analysis_type, location)
    except:
        continue
    properties = {}
    property_df = pd.DataFrame()
    for i in range(len(radiomics_results_df.keys())):
        key = list(radiomics_results_df.keys())[i]
        properties[i] = key
        print('properties ', i, ':', key)

    for i in range(len(properties)):
        selected_property = properties[i]

        property_result_df = pd.DataFrame.from_dict(radiomics_results_df[selected_property], 
                                                    orient = 'index').astype(float)

        result_df = analyze_result(overlaps, unet_df, property_result_df)
        result_df['property'] = selected_property
        if i != 0:
            result_df = result_df.drop(['WT dice', 'TC dice', 'ET dice'], axis = 0)
#         result_df.drop_duplicates(['bad_seg', 'good_seg'], inplace=True)
        property_df = pd.concat([property_df,result_df], axis = 0)
    property_df['analysis_type'] =  analysis_type  
    all_df = pd.concat([all_df,property_df], axis = 0)

properties  0 : original_shape_Elongation
properties  1 : original_shape_Flatness
properties  2 : original_shape_LeastAxisLength
properties  3 : original_shape_MajorAxisLength
properties  4 : original_shape_Maximum2DDiameterColumn
properties  5 : original_shape_Maximum2DDiameterRow
properties  6 : original_shape_Maximum2DDiameterSlice
properties  7 : original_shape_MinorAxisLength
properties  8 : original_shape_Sphericity
properties  0 : diagnostics_Mask-original_VoxelNum
properties  1 : diagnostics_Mask-original_VolumeNum
properties  2 : original_shape_MeshVolume
properties  3 : original_shape_SurfaceArea
properties  4 : original_shape_SurfaceVolumeRatio
properties  5 : original_shape_VoxelVolume


In [15]:
all_df = all_df.drop(['WT dice', 'TC dice', 'ET dice'], axis = 0)
all_df.to_csv('../Results/Analysis_Results/Radiomics/summary/all_shape_summary.csv', index=True)

In [18]:
radiomics_results_df['original_shape_VoxelVolume']

{'BraTS-GLI-00805-000': {'flair': 85169.0,
  't2': 85169.0,
  't1': 85169.0,
  't1ce': 85169.0},
 'BraTS-GLI-01496-000': {'flair': 218263.0,
  't2': 218263.0,
  't1': 218263.0,
  't1ce': 218263.0},
 'BraTS-GLI-01486-000': {'flair': 143067.0,
  't2': 143067.0,
  't1': 143067.0,
  't1ce': 143067.0},
 'BraTS-GLI-00645-001': {'flair': 20044.0,
  't2': 20044.0,
  't1': 20044.0,
  't1ce': 20044.0},
 'BraTS-GLI-00655-001': {'flair': 148517.0,
  't2': 148517.0,
  't1': 148517.0,
  't1ce': 148517.0},
 'BraTS-GLI-00382-000': {'flair': 53436.0,
  't2': 53436.0,
  't1': 53436.0,
  't1ce': 53436.0},
 'BraTS-GLI-00392-000': {'flair': 47590.0,
  't2': 47590.0,
  't1': 47590.0,
  't1ce': 47590.0},
 'BraTS-GLI-01064-000': {'flair': 33518.0,
  't2': 33518.0,
  't1': 33518.0,
  't1ce': 33518.0},
 'BraTS-GLI-00416-000': {'flair': 206933.0,
  't2': 206933.0,
  't1': 206933.0,
  't1ce': 206933.0},
 'BraTS-GLI-01074-000': {'flair': 95372.0,
  't2': 95372.0,
  't1': 95372.0,
  't1ce': 95372.0},
 'BraTS-GLI-00